# Synthetic Rare Disease Trio Variant Prioritization

I am analyzing a deterministic synthetic rare disease trio case. The proband has a suspected recessive neurodevelopmental disorder with optic atrophy and peripheral neuropathy, so the analysis focuses on rare functional variants inherited from both parents or present as homozygous alternate calls in the proband.

All variants below are synthetic and are generated inside this notebook.

In [2]:
import numpy as np
import pandas as pd

RNG_SEED = 240517
rng = np.random.default_rng(RNG_SEED)

PROJECT_ID = 'RDTRIO-SYN-042'
family = {
    'proband': 'RDTRIO-SYN-042-P1',
    'mother': 'RDTRIO-SYN-042-M1',
    'father': 'RDTRIO-SYN-042-F1',
}
phenotype_keywords = [
    'optic atrophy',
    'peripheral neuropathy',
    'cerebellar atrophy',
    'developmental delay',
    'hypotonia',
]

print(f'Project: {PROJECT_ID}')
print('Family samples: ' + ', '.join(f'{role}={sample}' for role, sample in family.items()))
print(f'Random seed fixed at {RNG_SEED}')
print('Phenotype keywords: ' + ', '.join(phenotype_keywords))


Project: RDTRIO-SYN-042
Family samples: proband=RDTRIO-SYN-042-P1, mother=RDTRIO-SYN-042-M1, father=RDTRIO-SYN-042-F1
Random seed fixed at 240517
Phenotype keywords: optic atrophy, peripheral neuropathy, cerebellar atrophy, developmental delay, hypotonia


## Synthetic Variant Table

The table includes trio genotypes, population allele frequency, consequence, pathogenicity score, and phenotype annotations. I deliberately include candidate genes `SLC25A46`, `PEX6`, and `ACADVL`, along with benign decoy genes and two records with incomplete annotation to exercise warning and exclusion behavior.

In [3]:
core_variants = [
    {
        'chromosome': '5', 'position': 110743221, 'gene': 'SLC25A46',
        'consequence': 'missense_variant', 'genotype_proband': '0/1',
        'genotype_mother': '0/1', 'genotype_father': '0/0',
        'population_af': 0.00018, 'pathogenicity_score': 0.91,
        'phenotype_terms': 'optic atrophy; peripheral neuropathy; cerebellar atrophy; developmental delay; hypotonia',
        'marker': 'SLC25A46_c.1019G>A_maternal', 'annotation_status': 'complete',
    },
    {
        'chromosome': '5', 'position': 110745902, 'gene': 'SLC25A46',
        'consequence': 'splice_donor_variant', 'genotype_proband': '0/1',
        'genotype_mother': '0/0', 'genotype_father': '0/1',
        'population_af': 0.00007, 'pathogenicity_score': 0.96,
        'phenotype_terms': 'optic atrophy; peripheral neuropathy; cerebellar atrophy; developmental delay; hypotonia',
        'marker': 'SLC25A46_c.1652+1G>T_paternal', 'annotation_status': 'complete',
    },
    {
        'chromosome': '6', 'position': 42938155, 'gene': 'PEX6',
        'consequence': 'frameshift_variant', 'genotype_proband': '0/1',
        'genotype_mother': '0/1', 'genotype_father': '0/0',
        'population_af': 0.00031, 'pathogenicity_score': 0.88,
        'phenotype_terms': 'peroxisomal biogenesis disorder; hypotonia; liver dysfunction',
        'marker': 'PEX6_c.2147del_maternal', 'annotation_status': 'complete',
    },
    {
        'chromosome': '6', 'position': 42941902, 'gene': 'PEX6',
        'consequence': 'missense_variant', 'genotype_proband': '0/0',
        'genotype_mother': '0/1', 'genotype_father': '0/1',
        'population_af': 0.00024, 'pathogenicity_score': 0.74,
        'phenotype_terms': 'peroxisomal biogenesis disorder; seizures',
        'marker': 'PEX6_c.1802A>G_absent', 'annotation_status': 'complete',
    },
    {
        'chromosome': '17', 'position': 7223941, 'gene': 'ACADVL',
        'consequence': 'missense_variant', 'genotype_proband': '1/1',
        'genotype_mother': '0/1', 'genotype_father': '0/1',
        'population_af': 0.00045, 'pathogenicity_score': 0.78,
        'phenotype_terms': 'cardiomyopathy; hypoketotic hypoglycemia; rhabdomyolysis; hypotonia',
        'marker': 'ACADVL_c.848T>C_homozygous', 'annotation_status': 'complete',
    },
    {
        'chromosome': '17', 'position': 7225120, 'gene': 'ACADVL',
        'consequence': 'synonymous_variant', 'genotype_proband': '0/1',
        'genotype_mother': '0/1', 'genotype_father': '0/0',
        'population_af': 0.00039, 'pathogenicity_score': 0.12,
        'phenotype_terms': 'fatty acid oxidation disorder',
        'marker': 'ACADVL_c.1113C>T_syn', 'annotation_status': 'complete',
    },
]

benign_decoy_genes = ['TTN', 'MUC16', 'GJB2', 'CFTR', 'RYR1']
benign_decoys = []
for index, gene in enumerate(benign_decoy_genes, start=1):
    benign_decoys.append({
        'chromosome': str(rng.choice(['1', '2', '7', '11', '19'])),
        'position': int(rng.integers(1_000_000, 190_000_000)),
        'gene': gene,
        'consequence': rng.choice(['intron_variant', 'synonymous_variant', 'missense_variant']),
        'genotype_proband': rng.choice(['0/1', '0/0']),
        'genotype_mother': rng.choice(['0/1', '0/0']),
        'genotype_father': rng.choice(['0/1', '0/0']),
        'population_af': round(float(rng.uniform(0.012, 0.080)), 5),
        'pathogenicity_score': round(float(rng.uniform(0.02, 0.35)), 2),
        'phenotype_terms': 'population polymorphism; unrelated trait',
        'marker': f'BENIGN_DECOY_{index:02d}_{gene}',
        'annotation_status': 'complete',
    })

missing_annotation_variants = [
    {
        'chromosome': '12', 'position': 55321044, 'gene': 'SYNMISSING1',
        'consequence': np.nan, 'genotype_proband': '0/1',
        'genotype_mother': '0/1', 'genotype_father': '0/0',
        'population_af': 0.00012, 'pathogenicity_score': 0.66,
        'phenotype_terms': 'optic atrophy', 'marker': 'SYN_MISSING_CONSEQUENCE_001',
        'annotation_status': 'missing_consequence',
    },
    {
        'chromosome': '3', 'position': 8821140, 'gene': 'SYNMISSING2',
        'consequence': 'missense_variant', 'genotype_proband': '0/1',
        'genotype_mother': '0/0', 'genotype_father': '0/1',
        'population_af': 0.00009, 'pathogenicity_score': np.nan,
        'phenotype_terms': np.nan, 'marker': 'SYN_MISSING_PHENOTYPE_002',
        'annotation_status': 'missing_score_and_phenotype',
    },
]

variant_table = pd.DataFrame(core_variants + benign_decoys + missing_annotation_variants)
display_columns = [
    'chromosome', 'position', 'gene', 'consequence', 'genotype_proband',
    'genotype_mother', 'genotype_father', 'population_af', 'pathogenicity_score', 'marker',
]

print(f'variant_table shape: {variant_table.shape}')
print('\nCore rare-disease genes represented:')
print(variant_table.loc[variant_table['gene'].isin(['SLC25A46', 'PEX6', 'ACADVL']), display_columns].to_string(index=False))


variant_table shape: (13, 12)

Core rare-disease genes represented:
chromosome  position     gene          consequence genotype_proband genotype_mother genotype_father  population_af  pathogenicity_score                        marker
         5 110743221 SLC25A46     missense_variant              0/1             0/1             0/0        0.00018                 0.91   SLC25A46_c.1019G>A_maternal
         5 110745902 SLC25A46 splice_donor_variant              0/1             0/0             0/1        0.00007                 0.96 SLC25A46_c.1652+1G>T_paternal
         6  42938155     PEX6   frameshift_variant              0/1             0/1             0/0        0.00031                 0.88       PEX6_c.2147del_maternal
         6  42941902     PEX6     missense_variant              0/0             0/1             0/1        0.00024                 0.74         PEX6_c.1802A>G_absent
        17   7223941   ACADVL     missense_variant              1/1             0/1             0/1   

In [4]:
required_annotation_columns = ['consequence', 'population_af', 'pathogenicity_score', 'phenotype_terms']
missing_annotation = variant_table[variant_table[required_annotation_columns].isna().any(axis=1)]

if not missing_annotation.empty:
    missing_markers = ', '.join(missing_annotation['marker'].tolist())
    import sys
    print(
        f'WARNING: {len(missing_annotation)} variants missing annotation and will be excluded from recessive filtering: {missing_markers}',
        file=sys.stderr,
    )


## Recessive Filtering Strategy

For this trio I apply a recessive model with three requirements: rare population allele frequency (`<= 0.001`), functional consequence, and a genotype pattern consistent with either compound heterozygosity or homozygous recessive inheritance. Singleton heterozygous alleles are retained only long enough to determine whether a second allele in the same gene completes the model.

In [5]:
DAMAGING_CONSEQUENCES = {
    'missense_variant',
    'splice_donor_variant',
    'splice_acceptor_variant',
    'frameshift_variant',
    'stop_gained',
}

def _infer_recessive_source(row):
    proband_gt = row['genotype_proband']
    mother_gt = row['genotype_mother']
    father_gt = row['genotype_father']

    if proband_gt == '1/1' and mother_gt == '0/1' and father_gt == '0/1':
        return 'both_parents_homozygous_model'
    if proband_gt == '0/1' and mother_gt == '0/1' and father_gt == '0/0':
        return 'maternal'
    if proband_gt == '0/1' and mother_gt == '0/0' and father_gt == '0/1':
        return 'paternal'
    return None

def apply_recessive_filter(variants, max_af=0.001):
    counts = [('input_variants', len(variants))]

    annotated = variants.dropna(subset=required_annotation_columns).copy()
    counts.append(('with_required_annotation', len(annotated)))

    rare = annotated[annotated['population_af'] <= max_af].copy()
    counts.append((f'rare_af_le_{max_af}', len(rare)))

    functional = rare[rare['consequence'].isin(DAMAGING_CONSEQUENCES)].copy()
    counts.append(('functional_consequence', len(functional)))

    functional['inherited_from'] = functional.apply(_infer_recessive_source, axis=1)
    genotype_pass = functional[functional['inherited_from'].notna()].copy()
    counts.append(('recessive_genotype_pattern', len(genotype_pass)))

    gene_model_rows = []
    for gene, group in genotype_pass.groupby('gene'):
        sources = set(group['inherited_from'])
        if 'both_parents_homozygous_model' in sources:
            gene_group = group[group['inherited_from'] == 'both_parents_homozygous_model'].copy()
            gene_group['gene_model'] = 'homozygous_recessive'
            gene_model_rows.append(gene_group)
        elif {'maternal', 'paternal'}.issubset(sources):
            gene_group = group[group['inherited_from'].isin(['maternal', 'paternal'])].copy()
            gene_group['gene_model'] = 'compound_heterozygous'
            gene_model_rows.append(gene_group)

    if gene_model_rows:
        model_consistent = pd.concat(gene_model_rows, ignore_index=True)
    else:
        model_consistent = genotype_pass.iloc[0:0].copy()
        model_consistent['gene_model'] = pd.Series(dtype='object')

    counts.append(('gene_model_consistent', len(model_consistent)))
    filter_counts = pd.DataFrame(counts, columns=['step', 'variant_count'])
    return model_consistent, filter_counts, genotype_pass

model_variants, filter_counts, genotype_pass_variants = apply_recessive_filter(variant_table)

print('Filter counts by step:')
print(filter_counts.to_string(index=False))


Filter counts by step:
                      step  variant_count
            input_variants             13
  with_required_annotation             11
          rare_af_le_0.001              6
    functional_consequence              5
recessive_genotype_pattern              4
     gene_model_consistent              3


In [6]:
def phenotype_match_score(row, observed_keywords):
    observed = {term.lower() for term in observed_keywords}
    annotated_terms = {
        term.strip().lower()
        for term in str(row['phenotype_terms']).split(';')
        if term.strip()
    }
    return round(len(observed.intersection(annotated_terms)) / len(observed), 2)

model_variants = model_variants.copy()
model_variants['phenotype_match_score'] = model_variants.apply(
    phenotype_match_score,
    axis=1,
    observed_keywords=phenotype_keywords,
)

model_strength = {'compound_heterozygous': 1.0, 'homozygous_recessive': 0.7}
gene_ranking = (
    model_variants.groupby(['gene', 'gene_model'], as_index=False)
    .agg(
        variant_count=('marker', 'size'),
        phenotype_match_score=('phenotype_match_score', 'max'),
        max_pathogenicity_score=('pathogenicity_score', 'max'),
        supporting_markers=('marker', lambda values: ', '.join(values)),
    )
)
gene_ranking['rank_score'] = (
    0.45 * gene_ranking['phenotype_match_score']
    + 0.45 * gene_ranking['max_pathogenicity_score']
    + 0.10 * gene_ranking['gene_model'].map(model_strength)
).round(2)
gene_ranking = gene_ranking.sort_values(['rank_score', 'gene'], ascending=[False, True]).reset_index(drop=True)

top_candidate_gene = gene_ranking.loc[0, 'gene']
compound_het_slc25a46 = model_variants[
    (model_variants['gene'] == 'SLC25A46')
    & (model_variants['gene_model'] == 'compound_heterozygous')
].copy()

candidate_columns = [
    'gene', 'marker', 'chromosome', 'position', 'consequence', 'inherited_from',
    'genotype_proband', 'genotype_mother', 'genotype_father', 'population_af',
    'pathogenicity_score', 'phenotype_match_score',
]
ranking_columns = [
    'gene', 'gene_model', 'variant_count', 'phenotype_match_score',
    'max_pathogenicity_score', 'rank_score', 'supporting_markers',
]

print('Gene ranking by phenotype-aware score:')
print(gene_ranking[ranking_columns].to_string(index=False))
print('\nCompound heterozygous SLC25A46 candidate variants:')
print(compound_het_slc25a46[candidate_columns].to_string(index=False))
print(f'\ntop_candidate_gene {top_candidate_gene}')


Gene ranking by phenotype-aware score:
    gene            gene_model  variant_count  phenotype_match_score  max_pathogenicity_score  rank_score                                         supporting_markers
SLC25A46 compound_heterozygous              2                    1.0                     0.96        0.98 SLC25A46_c.1019G>A_maternal, SLC25A46_c.1652+1G>T_paternal
  ACADVL  homozygous_recessive              1                    0.2                     0.78        0.51                                 ACADVL_c.848T>C_homozygous

Compound heterozygous SLC25A46 candidate variants:
    gene                        marker chromosome  position          consequence inherited_from genotype_proband genotype_mother genotype_father  population_af  pathogenicity_score  phenotype_match_score
SLC25A46   SLC25A46_c.1019G>A_maternal          5 110743221     missense_variant       maternal              0/1             0/1             0/0        0.00018                 0.91                    1.0
SLC25A

## Interpretation

I prioritize `SLC25A46` because the proband carries two rare functional variants in trans: a maternally inherited missense variant and a paternally inherited splice donor variant. Both variants are absent from the unaffected parent on the opposite haplotype, both are below the population allele-frequency threshold, and the gene-level phenotype annotation directly matches the observed optic atrophy, peripheral neuropathy, cerebellar atrophy, developmental delay, and hypotonia.

`PEX6` contributes only one inherited rare functional allele in this synthetic trio, so it does not complete the recessive model. `ACADVL` has a homozygous recessive genotype pattern, but its phenotype terms are dominated by fatty-acid oxidation and cardiomyopathy features, making it a weaker match to the proband's neuro-ophthalmic presentation.

In [7]:
selected_gene_row = gene_ranking.loc[gene_ranking['gene'] == top_candidate_gene].iloc[0]
selected_markers = '; '.join(compound_het_slc25a46['marker'].tolist())

summary_report = {
    'project_id': PROJECT_ID,
    'top_candidate_gene': top_candidate_gene,
    'inheritance_model': selected_gene_row['gene_model'],
    'selected_markers': selected_markers,
    'final_rank_score': f"{selected_gene_row['rank_score']:.2f}",
    'conclusion': 'SLC25A46 is the highest-priority synthetic candidate for the suspected recessive rare disease trio.',
}

for metric, value in summary_report.items():
    print(f'{metric}: {value}')


project_id: RDTRIO-SYN-042
top_candidate_gene: SLC25A46
inheritance_model: compound_heterozygous
selected_markers: SLC25A46_c.1019G>A_maternal; SLC25A46_c.1652+1G>T_paternal
final_rank_score: 0.98
conclusion: SLC25A46 is the highest-priority synthetic candidate for the suspected recessive rare disease trio.
